# Project AGASTYA (SIH26168)
## Objective 5: Causal Residual Learning Model Training & Validation
**Platform:** Google Colab / PyTorch  
**Purpose:** Train and validate the first causal neural residual estimator (`CausalResidualGRU`) for hybrid dead reckoning.


In [ ]:
# ============================================================
# 01_environment_setup
# ============================================================
import os
import sys
import json
import random
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Model:', torch.cuda.get_device_name(0))


In [ ]:
# ============================================================
# 02_repository_setup & Google Drive Mounting
# ============================================================
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/AGASTYA'
except Exception:
    PROJECT_ROOT = os.path.abspath(os.getcwd())

print('Configured PROJECT_ROOT:', PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)


In [ ]:
# ============================================================
# 03_dataset_discovery
# ============================================================
processed_dir = os.path.join(PROJECT_ROOT, 'data', 'processed', 'sequences')
train_seq = 'sync_01'
val_seq = 'v_standalone_03'
test_seq = 'sync_02'

print('Available Sequences:')
for s in [train_seq, val_seq, test_seq]:
    p = os.path.join(processed_dir, s, 'navigation_inputs.parquet')
    print(f'  * {s}: exists={os.path.exists(p)}')


In [ ]:
# ============================================================
# 04_feature_validation & Canonical Registry
# ============================================================
from src.ai_residual.feature_registry import CANONICAL_FEATURE_NAMES, validate_feature_matrix_columns

print(f'Canonical Causal Features ({len(CANONICAL_FEATURE_NAMES)} total):')
for i, name in enumerate(CANONICAL_FEATURE_NAMES):
    print(f'  [{i:02d}] {name}')

validate_feature_matrix_columns(CANONICAL_FEATURE_NAMES)
print('Feature schema validation: PASSED')


In [ ]:
# ============================================================
# 05_target_generation & 06_train_val_test_split
# ============================================================
from scripts.train_residual_model import prepare_sequence_data

print('Extracting causal features and residual targets...')
proc_base = os.path.join(PROJECT_ROOT, 'data', 'processed')
train_data = prepare_sequence_data(train_seq, proc_base)
val_data = prepare_sequence_data(val_seq, proc_base)
test_data = prepare_sequence_data(test_seq, proc_base)

print('Train samples:', len(train_data['causal_feats_df']))
print('Val samples:  ', len(val_data['causal_feats_df']))
print('Test samples: ', len(test_data['causal_feats_df']), '(HELD-OUT UNSEEN)')


In [ ]:
# ============================================================
# 07_training_scalers (Fitted STRICTLY on sync_01)
# ============================================================
from src.ai_residual.scaler import TrainOnlyScaler, TargetScaler

feat_scaler = TrainOnlyScaler().fit(train_data['causal_feats_df'], sequence_id=train_seq)
target_scaler = TargetScaler().fit(train_data['targets_matrix'], sequence_id=train_seq)

artifacts_dir = os.path.join(PROJECT_ROOT, 'artifacts', 'objective5')
os.makedirs(artifacts_dir, exist_ok=True)
feat_scaler.save_json(os.path.join(artifacts_dir, 'feature_scaler.json'))
target_scaler.save_json(os.path.join(artifacts_dir, 'target_scaler.json'))
print('Train-only scalers saved to artifacts/objective5/')


In [ ]:
# ============================================================
# 08_causal_window_generation (W = 10 epochs ~ 1.0s)
# ============================================================
from src.ai_residual.dataset import CausalWindowDataset

w_size = 10
train_ds = CausalWindowDataset(feat_scaler.transform(train_data['causal_feats_df']), target_scaler.transform(train_data['targets_matrix']), window_size=w_size)
val_ds = CausalWindowDataset(feat_scaler.transform(val_data['causal_feats_df']), target_scaler.transform(val_data['targets_matrix']), window_size=w_size)
test_ds = CausalWindowDataset(feat_scaler.transform(test_data['causal_feats_df']), target_scaler.transform(test_data['targets_matrix']), window_size=w_size)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

print(f'Causal Windows: Train={len(train_ds)}, Val={len(val_ds)}, Test={len(test_ds)}')


In [ ]:
# ============================================================
# 09_model_definition (CausalResidualGRU)
# ============================================================
from src.ai_residual.model import CausalResidualGRU

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CausalResidualGRU(input_dim=16, hidden_dim=64, mlp_dim=32, output_dim=2).to(device)
print(model)
print('Total trainable parameters:', sum(p.numel() for p in model.parameters()))


In [ ]:
# ============================================================
# 10_training & 11_validation (Early Stopping on Val Loss)
# ============================================================
from src.ai_residual.trainer import ResidualModelTrainer, set_seed

set_seed(42)
trainer = ResidualModelTrainer(model=model, learning_rate=1e-3, device=device)
train_result = trainer.fit(train_loader, val_loader, max_epochs=100, patience=15, checkpoint_dir=artifacts_dir)
print(f'Best Epoch: {train_result["best_epoch"]} | Best Val Loss: {train_result["best_val_loss"]:.6f}')


In [ ]:
# ============================================================
# 12_checkpoint_selection & 13_held_out_test
# ============================================================
from src.ai_residual.evaluator import ResidualEvaluator

res_metrics, y_true_p, y_pred_p, t_stamps = ResidualEvaluator.evaluate_dataset(
    model, test_loader, target_scaler, device=device
)
for k, v in res_metrics.items():
    print(f'{k}: MAE={v.mae:.5f} | RMSE={v.rmse:.5f} | Bias={v.bias:+.5f} | R2={v.r2_score:.4f} | r={v.pearson_correlation:+.4f}')


In [ ]:
# ============================================================
# 14_navigation_rollout & 15_baseline_comparison
# ============================================================
from src.ai_residual.rollout import AIRolloutEngine
from navigation_engine.evaluation import DeadReckoningEvaluator

rollout_engine = AIRolloutEngine(model=model, feature_scaler=feat_scaler, target_scaler=target_scaler, device=device)
ai_traj = rollout_engine.run_rollout(
    test_data['nav_df'], test_data['causal_feats_df'],
    initial_p_east_m=float(test_data['ref_df']['pos_east_m'].iloc[0]),
    initial_p_north_m=float(test_data['ref_df']['pos_north_m'].iloc[0]),
    initial_heading_rad=float(test_data['ref_df']['heading_rad'].iloc[0])
)

ref_e = test_data['ref_df']['pos_east_m'].to_numpy()
ref_n = test_data['ref_df']['pos_north_m'].to_numpy()
ref_h = test_data['ref_df'].get('heading_rad', None)
ref_v = test_data['ref_df'].get('ground_speed_ms', None)

c_m, _, _ = DeadReckoningEvaluator.evaluate(test_data['classical_traj'], ref_e, ref_n, ref_h, ref_v)
ai_m, _, _ = DeadReckoningEvaluator.evaluate(ai_traj, ref_e, ref_n, ref_h, ref_v)

print(f'Classical Baseline A ATE RMSE: {c_m.ate_rmse_m:.4f} m | Final Error: {c_m.final_position_error_m:.4f} m')
print(f'AI-Corrected Baseline ATE RMSE: {ai_m.ate_rmse_m:.4f} m | Final Error: {ai_m.final_position_error_m:.4f} m')


In [ ]:
# ============================================================
# 16_diagnostics & 17_artifact_export & 18_reproducibility_check
# ============================================================
from src.ai_residual.diagnostics import Objective5Visualizer
from src.ai_residual.outage_eval import OutageComparator
from src.ai_residual.ablations import AblationRunner

outage_recs = OutageComparator.evaluate_outages(test_data['classical_traj'], ai_traj, ref_e, ref_n)
abl_recs = AblationRunner.run_ablations(model, feat_scaler, target_scaler, test_data['nav_df'], test_data['causal_feats_df'], test_data['ref_df'], device=device)

fig_dir = os.path.join(artifacts_dir, 'figures')
figs = Objective5Visualizer.generate_all_plots(
    train_result, y_true_p, y_pred_p, t_stamps,
    test_data['classical_traj'], ai_traj,
    ref_e, ref_n,
    ref_v.to_numpy() if ref_v is not None else None,
    ref_h.to_numpy() if ref_h is not None else None,
    outage_recs, abl_recs,
    output_dir=fig_dir
)

print(f'Successfully rendered all 12 diagnostic figures to {fig_dir}')
print('\n' + '=' * 60)
print('OBJECTIVE 5 TRAINING COMPLETE')
print('=' * 60)
